In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [5]:
llm = ChatGroq(temperature=0, groq_api_key="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", model_name="llama-3.3-70b-versatile")


In [6]:
res=llm.invoke("The first person to land on moon is?")

In [7]:
res.content

'The first person to land on the moon is Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon\'s surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That\'s one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.'

In [14]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
import torch

# Load the persisted ChromaDB store using the same model as before
EMBED_MODEL = "hkunlp/instructor-xl"
embedding = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Load Chroma vector store
db = Chroma(persist_directory="chroma_store", embedding_function=embedding)

def query_chroma_db(query):
    # Apply instruction-based input formatting
    formatted_query = f"Represent the hotel booking question: {query}"
    print(formatted_query)
    
    # Perform similarity search (no need to manually call embed_query)
    results = db.similarity_search(formatted_query, k=3)

    # Combine the top results into a single context
    context = "\n\n".join([result.page_content for result in results])

    return context


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


In [9]:
from langchain.prompts import ChatPromptTemplate

# Create a tailored prompt
def create_prompt(query, context):
    prompt = f"""
    Based on the following booking data context, answer the question:

    Context:
    {context}

    Question: {query}

    Answer:
    """
    return prompt



In [12]:
from langchain_groq import ChatGroq


def get_answer_from_llm(query):
    # Step 1: Query ChromaDB to get the context
    context = query_chroma_db(query)
    print(context)
    
    # Step 2: Create a tailored prompt using the context and the query
    prompt = create_prompt(query, context)
    
    # Step 3: Use LLM to generate an answer based on the prompt
    response = llm.invoke(prompt)
    
    return response


In [15]:
query = "Show me total revenue for June 2017"
answer = get_answer_from_llm(query)
print(f"Answer: {answer.content}")


Represent the hotel booking question: Show me total revenue for June 2017

Answer: To provide an accurate answer, I would need the actual booking data for June 2017, which is not provided in your query. However, I can guide you through a general approach to calculate the total revenue for June 2017 based on hypothetical booking data.

1. **Identify Relevant Data**: You would need a dataset that includes at least the following columns:
   - Booking Date
   - Revenue (or a way to calculate it, such as room rate and number of nights stayed)

2. **Filter Data for June 2017**: From your dataset, filter the bookings to only include those made in June 2017. This can typically be done using a date filter in your data analysis tool or programming language.

3. **Calculate Total Revenue**: Once you have the filtered data, you can calculate the total revenue by summing up the revenue from each booking. If your data includes the cost per night and the number of nights, you would multiply these two

In [ ]:
import pandas as pd


In [19]:
for _, row in df.iterrows():
    if row["arrival_date_month"] == "June" and row["arrival_date_year"] == 2017:
        print(f"✅ Found: Hotel: {row['hotel']}, ADR: {row['adr']}, Month: {row['arrival_date_month']}, Year: {row['arrival_date_year']}")


NameError: name 'df' is not defined

In [18]:
import requests

res = requests.post("http://127.0.0.1:8000/ask", json={"query": "Which locations had the highest booking cancellations?"})
print(res.json())


{'response': 'Based on the provided data, it appears that the USA (specifically, the City Hotel in the USA) had the highest number of booking cancellations, with two cancellations in June 2016.'}


In [19]:
res.content

b'{"response":"Based on the provided data, it appears that the USA (specifically, the City Hotel in the USA) had the highest number of booking cancellations, with two cancellations in June 2016."}'